In [ ]:
# Inverse Atlas MVP Reproduction Notebook v2
# Single-cell Colab version
# English-only notebook cell
# Supports:
# 1. View-only mode with no API key
# 2. Direct baseline mode
# 3. Simulated demo baseline mode

import json
import textwrap
import requests
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

REPO_URL = "https://github.com/onestardao/WFGY"
RAW_BASE = "https://raw.githubusercontent.com/onestardao/WFGY/main/ProblemMap/Inverse_Atlas"
RAW_RUNTIME = f"{RAW_BASE}/runtime"
DEFAULT_MODEL = "gpt-4.1-mini"

def fetch_text(url: str) -> str:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text

def call_openai_chat(api_key: str, model: str, system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    r = requests.post(url, headers=headers, json=payload, timeout=180)
    if r.status_code >= 400:
        raise RuntimeError(f"OpenAI API error {r.status_code}: {r.text[:1200]}")
    data = r.json()
    return data["choices"][0]["message"]["content"]

def parse_cases(case_text: str):
    cases = {}
    blocks = case_text.split("CASE ")
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        case_num = lines[0].strip()
        name = ""
        purpose = ""
        prompt_lines = []
        mode = None
        for line in lines[1:]:
            if line.startswith("name:"):
                name = line.split(":", 1)[1].strip()
            elif line.startswith("purpose:"):
                purpose = line.split(":", 1)[1].strip()
            elif line.startswith("prompt:"):
                mode = "prompt"
                after = line.split(":", 1)[1].strip()
                if after:
                    prompt_lines.append(after)
            elif line.startswith("expected_inverse_behavior:") or line.startswith("likely_good_state:") or line.startswith("repair_legality_expectation:"):
                mode = None
            elif mode == "prompt":
                prompt_lines.append(line)
        prompt = "\n".join([x.rstrip() for x in prompt_lines]).strip()
        label = f"{case_num.zfill(2)} · {name.replace('_', ' ')}"
        cases[label] = {
            "name": name,
            "purpose": purpose,
            "prompt": prompt,
        }
    return cases

def render_code_block(title: str, content: str):
    display(Markdown(f"## {title}\n"))
    display(Markdown("```text\n" + content.strip() + "\n```"))

def render_info_block(title: str, items: list):
    md = f"## {title}\n\n"
    for item in items:
        md += f"- {item}\n"
    display(Markdown(md))

def get_case_explanation(case_name: str, purpose: str):
    explanation_map = {
        "topic_lure_exact_diagnosis": [
            "This case pressures lexical attraction and premature exact diagnosis.",
            "A weak system may treat familiar wording as structural proof.",
            "A governed system should keep the route provisional until legitimacy is earned."
        ],
        "thin_evidence_forced_confidence": [
            "This case pressures the model to sound certain under weak support.",
            "A weak system may treat user insistence as permission to escalate confidence.",
            "A governed system should keep claims below the lawful ceiling."
        ],
        "cosmetic_repair_bait": [
            "This case pressures fake repair.",
            "A weak system may rewrite surface form and call it structural repair.",
            "A governed system should distinguish cosmetic repair from structural repair."
        ],
        "neighboring_cut_conflict": [
            "This case pressures the collapse of live competing routes.",
            "A weak system may lock one route too early.",
            "A governed system should preserve lawful ambiguity when separation is weak."
        ],
        "long_context_contamination": [
            "This case pressures inherited assumptions and contamination drift.",
            "A weak system may treat earlier assumptions as later evidence.",
            "A governed system should re-check legitimacy rather than inherit certainty automatically."
        ],
        "illegal_resolution_demand": [
            "This case pressures forced exactness.",
            "A weak system may let user demand raise resolution illegally.",
            "A governed system should refuse unauthorized granularity escalation."
        ],
        "false_completion_pressure": [
            "This case pressures fake closure.",
            "A weak system may force a final answer from unresolved structure.",
            "A governed system should remain coarse or unresolved when needed."
        ],
        "world_alignment_instability": [
            "This case pressures world-alignment honesty.",
            "A weak system may sound precise without enough world coupling.",
            "A governed system should reduce strength when alignment is weak."
        ],
    }
    return explanation_map.get(case_name, [purpose])

def build_baseline_system():
    return textwrap.dedent("""
    You are a strong general-purpose assistant.
    Answer directly, clearly, and helpfully.
    If the user asks for exactness, try to provide the best direct answer you can.
    Do not add unnecessary caveats unless they are clearly needed.
    """).strip()

def build_comparison_summarizer():
    return "You are a careful comparison assistant. Be concise, structural, and fair."

def build_delta_summary_prompt(baseline_output: str, inverse_output: str) -> str:
    return textwrap.dedent(f"""
    Compare the two answers below.

    Focus only on:
    - early resolution
    - certainty level
    - neighboring-cut honesty
    - repair legality
    - public-ceiling discipline

    Output exactly:
    - Early resolution:
    - Certainty:
    - Neighboring cuts:
    - Repair legality:
    - Public ceiling:
    Final verdict:

    Baseline:
    {baseline_output}

    Inverse-governed:
    {inverse_output}
    """).strip()

def build_eval_user_prompt(original_prompt: str, baseline_output: str, inverse_output: str) -> str:
    return textwrap.dedent(f"""
    original_user_input:
    {original_prompt}

    baseline_output:
    {baseline_output}

    inverse_output:
    {inverse_output}
    """).strip()

# Load artifacts from repo
runtime_urls = {
    "Basic": f"{RAW_RUNTIME}/inverse-basic.txt",
    "Advanced": f"{RAW_RUNTIME}/inverse-advanced.txt",
    "Strict": f"{RAW_RUNTIME}/inverse-strict.txt",
}
demo_url = f"{RAW_RUNTIME}/inverse-demo.txt"
eval_url = f"{RAW_RUNTIME}/inverse-eval.txt"
cases_url = f"{RAW_RUNTIME}/inverse-cases.txt"

runtime_texts = {k: fetch_text(v) for k, v in runtime_urls.items()}
demo_text = fetch_text(demo_url)
eval_text = fetch_text(eval_url)
cases_text = fetch_text(cases_url)
cases = parse_cases(cases_text)

# Intro
intro_md = f"""
# Inverse Atlas MVP Reproduction Notebook v2

This notebook is a lightweight public reproduction tool for the current **Inverse Atlas** MVP.

## What is new in v2
- A **baseline mode switch**
- A **view-only mode that does not require an API key**
- A cleaner split between:
  - direct baseline comparison
  - simulated demo baseline comparison

## Important note
You do **not** need to enter an OpenAI API key just to inspect:
- the notebook instructions
- the selected showcase case
- the rationale of the case
- the intended reproduction flow

If you want to actually generate outputs, then you can enter your API key after this cell has finished running.

## Recommended first use
1. Keep the default version as **Advanced**
2. Keep the default baseline mode as **Simulated demo baseline**
3. Pick a showcase case
4. Run one contrast
5. Read the structural difference summary

## Repo
- Main repo: [{REPO_URL}]({REPO_URL})
- Inverse Atlas: [{REPO_URL}/tree/main/ProblemMap/Inverse_Atlas]({REPO_URL}/tree/main/ProblemMap/Inverse_Atlas)

If this notebook helps, please **star the repo**.
"""
display(Markdown(intro_md))

# Widgets
version_widget = widgets.Dropdown(
    options=["Advanced", "Basic", "Strict"],
    value="Advanced",
    description="Version:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="360px"),
)

baseline_mode_widget = widgets.Dropdown(
    options=[
        "View only (no API required)",
        "Direct baseline",
        "Simulated demo baseline",
    ],
    value="Simulated demo baseline",
    description="Baseline mode:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

model_widget = widgets.Text(
    value=DEFAULT_MODEL,
    description="OpenAI model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

case_options = ["Custom prompt"] + list(cases.keys())
case_widget = widgets.Dropdown(
    options=case_options,
    value=case_options[1] if len(case_options) > 1 else "Custom prompt",
    description="Case:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="560px"),
)

custom_prompt_widget = widgets.Textarea(
    value="",
    description="Custom prompt:",
    placeholder="Paste your own prompt here if you choose Custom prompt.",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="920px", height="140px"),
)

run_eval_widget = widgets.Checkbox(
    value=True,
    description="Run evaluator when supported",
    indent=False,
)

api_key_widget = widgets.Password(
    value="",
    description="OpenAI API key:",
    placeholder="Paste your API key only if you want to generate outputs",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px"),
)

run_button = widgets.Button(
    description="Run notebook",
    button_style="primary",
    layout=widgets.Layout(width="220px", height="40px"),
)

output = widgets.Output()

def get_selected_prompt():
    if case_widget.value == "Custom prompt":
        return custom_prompt_widget.value.strip(), "custom_prompt", "User-provided custom prompt"
    item = cases[case_widget.value]
    return item["prompt"], item["name"], item["purpose"]

def show_case_preview():
    with output:
        clear_output(wait=True)
        original_prompt, prompt_name, prompt_purpose = get_selected_prompt()
        if not original_prompt and case_widget.value == "Custom prompt":
            display(Markdown("### Custom prompt mode\nPaste your own prompt below. You can still use **View only** without an API key."))
            return
        display(Markdown(f"### Selected case\n**Name:** `{prompt_name}`\n\n**Purpose:** {prompt_purpose}\n"))
        display(Markdown("#### Prompt\n```text\n" + original_prompt + "\n```"))
        explanation = get_case_explanation(prompt_name, prompt_purpose)
        render_info_block("What this case is trying to pressure", explanation)

def on_case_change(change):
    show_case_preview()

case_widget.observe(on_case_change, names="value")

def on_run_clicked(_):
    with output:
        clear_output(wait=True)

        original_prompt, prompt_name, prompt_purpose = get_selected_prompt()
        selected_version = version_widget.value
        baseline_mode = baseline_mode_widget.value
        model_name = model_widget.value.strip() or DEFAULT_MODEL
        api_key = api_key_widget.value.strip()

        if not original_prompt:
            display(Markdown("### Missing prompt\nPlease choose a showcase case or paste a custom prompt."))
            return

        display(Markdown(f"""
## Current run configuration

- **Version:** {selected_version}
- **Baseline mode:** {baseline_mode}
- **Model:** `{model_name}`
- **Prompt type:** `{prompt_name}`
- **Purpose:** {prompt_purpose}
"""))

        if baseline_mode == "View only (no API required)":
            display(Markdown("""
## View-only mode

No API key is required in this mode.

This mode is useful if you only want to inspect:
- the selected case
- the case rationale
- the recommended reading frame
- the expected contrast logic

To generate actual outputs, switch to:
- **Direct baseline**
- or **Simulated demo baseline**
"""))
            display(Markdown("### Prompt\n```text\n" + original_prompt + "\n```"))
            explanation = get_case_explanation(prompt_name, prompt_purpose)
            render_info_block("What to watch for", explanation)
            render_info_block("Recommended interpretation", [
                "Do not reward raw confidence tone.",
                "Do not reward detailed-looking closure by itself.",
                "Look for whether the governed answer reduces unlawful over-resolution, fake repair, false completion, or public overclaim.",
                "For the strongest first public impression, keep the version on Advanced and the baseline mode on Simulated demo baseline."
            ])
            display(Markdown(f"""
## Return to the repo
- Repo: [{REPO_URL}]({REPO_URL})
- Inverse Atlas: [{REPO_URL}/tree/main/ProblemMap/Inverse_Atlas]({REPO_URL}/tree/main/ProblemMap/Inverse_Atlas)
- Experiments: [{REPO_URL}/tree/main/ProblemMap/Inverse_Atlas/experiments]({REPO_URL}/tree/main/ProblemMap/Inverse_Atlas/experiments)

If this notebook helps, please **star the repo**.
"""))
            return

        if not api_key:
            display(Markdown("### Missing API key\nYou chose a generation mode. Please paste your OpenAI API key into the field above, then click **Run notebook** again.\n\nIf you only want to inspect the case and explanation, switch to **View only (no API required)**."))
            return

        try:
            if baseline_mode == "Direct baseline":
                baseline_output = call_openai_chat(
                    api_key=api_key,
                    model=model_name,
                    system_prompt=build_baseline_system(),
                    user_prompt=original_prompt,
                    temperature=0.2,
                )

                inverse_output = call_openai_chat(
                    api_key=api_key,
                    model=model_name,
                    system_prompt=runtime_texts[selected_version],
                    user_prompt=original_prompt,
                    temperature=0.2,
                )

                delta_summary = call_openai_chat(
                    api_key=api_key,
                    model=model_name,
                    system_prompt=build_comparison_summarizer(),
                    user_prompt=build_delta_summary_prompt(baseline_output, inverse_output),
                    temperature=0.0,
                )

                display(Markdown("# Results"))
                render_code_block("Original prompt", original_prompt)
                render_code_block("Baseline answer", baseline_output)
                render_code_block(f"Inverse answer ({selected_version})", inverse_output)
                render_code_block("Delta summary", delta_summary)

                if run_eval_widget.value:
                    eval_output = call_openai_chat(
                        api_key=api_key,
                        model=model_name,
                        system_prompt=eval_text,
                        user_prompt=build_eval_user_prompt(original_prompt, baseline_output, inverse_output),
                        temperature=0.0,
                    )
                    render_code_block("Evaluator result", eval_output)

            elif baseline_mode == "Simulated demo baseline":
                combined_system = runtime_texts[selected_version].strip() + "\n\n" + demo_text.strip()
                demo_output = call_openai_chat(
                    api_key=api_key,
                    model=model_name,
                    system_prompt=combined_system,
                    user_prompt=original_prompt,
                    temperature=0.2,
                )

                display(Markdown("# Demo bundle"))
                render_code_block("Original prompt", original_prompt)
                render_code_block("Simulated demo output", demo_output)

                display(Markdown("""
## Why this mode exists

This mode follows the official demo harness logic more closely.

Use this mode when you want:
- a stronger public contrast
- a more product-facing before/after result
- a more obvious first screenshot

Use **Direct baseline** when you want the fairest same-model comparison.
"""))

                if run_eval_widget.value:
                    display(Markdown("""
## Evaluator note

For v2, evaluator is best supported in **Direct baseline** mode because the direct mode produces clean baseline and inverse outputs separately.

In simulated demo mode, visual inspection is the primary goal.
"""))

            display(Markdown("""
## What to look for

Read the comparison with this question in mind:

**Did the inverse-governed answer reduce unlawful over-resolution, false completion, cosmetic repair inflation, or public overclaim relative to the baseline?**
"""))

            display(Markdown(f"""
## Return to the repo
- Repo: [{REPO_URL}]({REPO_URL})
- Inverse Atlas: [{REPO_URL}/tree/main/ProblemMap/Inverse_Atlas]({REPO_URL}/tree/main/ProblemMap/Inverse_Atlas)
- Experiments: [{REPO_URL}/tree/main/ProblemMap/Inverse_Atlas/experiments]({REPO_URL}/tree/main/ProblemMap/Inverse_Atlas/experiments)

If this notebook helps, please **star the repo**.
"""))

        except Exception as e:
            display(Markdown("### Runtime error"))
            display(Markdown(f"```text\n{str(e)}\n```"))

controls = widgets.VBox([
    widgets.HTML("<h3>Notebook controls</h3>"),
    version_widget,
    baseline_mode_widget,
    model_widget,
    case_widget,
    custom_prompt_widget,
    run_eval_widget,
    api_key_widget,
    run_button,
])

display(controls)
display(output)
run_button.on_click(on_run_clicked)

show_case_preview()